# 🏠 House Price Prediction Project
> **Objective**: Building a Linear Regression model to estimate property values using the USA Housing dataset.
---

In [43]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from xgboost import XGBRegressor

In [44]:
df = pd.read_csv("data.csv")

In [45]:
df.columns

Index(['date', 'price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot',
       'floors', 'waterfront', 'view', 'condition', 'sqft_above',
       'sqft_basement', 'yr_built', 'yr_renovated', 'street', 'city',
       'statezip', 'country'],
      dtype='object')

In [46]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4600 entries, 0 to 4599
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   date           4600 non-null   object 
 1   price          4600 non-null   float64
 2   bedrooms       4600 non-null   float64
 3   bathrooms      4600 non-null   float64
 4   sqft_living    4600 non-null   int64  
 5   sqft_lot       4600 non-null   int64  
 6   floors         4600 non-null   float64
 7   waterfront     4600 non-null   int64  
 8   view           4600 non-null   int64  
 9   condition      4600 non-null   int64  
 10  sqft_above     4600 non-null   int64  
 11  sqft_basement  4600 non-null   int64  
 12  yr_built       4600 non-null   int64  
 13  yr_renovated   4600 non-null   int64  
 14  street         4600 non-null   object 
 15  city           4600 non-null   object 
 16  statezip       4600 non-null   object 
 17  country        4600 non-null   object 
dtypes: float

In [47]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from catboost import CatBoostRegressor

# Load dataset
df = pd.read_csv("data.csv")

# 1. CLEANING BASIC
# Filter harga 0 dan buang kolom yang gak ngaruh/noise
df = df[df['price'] > 0]
df = df.drop(columns=['date', 'street', 'country', 'waterfront'])

# 2. PARSING ZIP
# Ekstrak kode pos dari statezip
df[['state', 'zip']] = df['statezip'].str.split(r'\s+', expand=True)
df = df.drop(columns=['statezip', 'state'])

# 3. FEATURE ENGINEERING
# Ubah tahun jadi umur rumah (asumsi data 2014)
df['house_age'] = 2014 - df['yr_built']
df = df.drop(columns=['yr_built', 'yr_renovated'])

# 4. OUTLIER & SKEWNESS
# Buang 1% harga paling ga ngotak (mansion/istana)
limit = df["price"].quantile(0.99)
df = df[df["price"] < limit]

# Log transform luas tanah dan harga biar distribusinya waras
df['sqft_lot'] = np.log1p(df['sqft_lot'])
df['price'] = np.log1p(df['price'])

# 5. SPLIT DATA
X = df.drop('price', axis=1)
y = df['price']

# WAJIB: Pastikan kolom fitur lokasi tipenya string (objek) biar dibaca sebagai kategori
X['city'] = X['city'].astype(str)
X['zip'] = X['zip'].astype(str)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 6. DAFTARKAN FITUR KATEGORI LOKASI
cat_features = ['city', 'zip']

# 7. MODELING PAKE CATBOOST
model = CatBoostRegressor(
    iterations=1500,         # Jumlah pohon
    learning_rate=0.05,      # Kecepatan belajar
    depth=7,                 # Kedalaman pohon
    eval_metric='RMSE',
    cat_features=cat_features, # Masukin list lokasi tadi ke sini!
    random_seed=42,
    verbose=100              # Cuma nge-print tiap 100 iterasi biar terminal lu ga penuh
)

# Fit model (pakai eval_set buat ngecek overfitting otomatis)
model.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=50)

# 8. PREDIKSI & EVALUASI
y_pred = model.predict(X_test)

# Kembalikan skala log ke harga asli (Ori)
y_pred_ori = np.expm1(y_pred)
y_test_ori = np.expm1(y_test)

# Hitung R2 dan RMSE
r2_log = r2_score(y_test, y_pred)
r2_ori = r2_score(y_test_ori, y_pred_ori)
rmse_log = np.sqrt(mean_squared_error(y_test, y_pred))
rmse_ori = np.sqrt(mean_squared_error(y_test_ori, y_pred_ori))

print("="*30)
print(f"R2 log   = {r2_log:.4f}")
print(f"R2 ori   = {r2_ori:.4f}")
print(f"RMSE log = {rmse_log:.4f}")
print(f"RMSE ori = {rmse_ori:.2f}")
print("="*30)

0:	learn: 0.5019577	test: 0.4773405	best: 0.4773405 (0)	total: 294ms	remaining: 7m 20s
100:	learn: 0.2202813	test: 0.2331575	best: 0.2331575 (100)	total: 11.3s	remaining: 2m 36s
200:	learn: 0.1995015	test: 0.2291273	best: 0.2291273 (200)	total: 21.2s	remaining: 2m 17s
300:	learn: 0.1826298	test: 0.2288296	best: 0.2284985 (275)	total: 30.3s	remaining: 2m
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.2284985217
bestIteration = 275

Shrink model to first 276 iterations.
R2 log   = 0.7845
R2 ori   = 0.7678
RMSE log = 0.2285
RMSE ori = 134508.79
